In [12]:
import stimwrap as st


In [43]:
st.resave(input="STIM_inputs/ref_cell_positions.csv,STIM_inputs/ref_expression.csv,ref.h5ad", container="heart_og.n5")
st.resave(input="STIM_inputs/query_cell_positions.csv,STIM_inputs/query_expression.csv,query.h5ad", container="heart_og.n5")

[2025-08-26T17:14:21,434] [main] INFO : Log level set to: INFO
[2025-08-26T17:14:21,880] [main] INFO : Saving in file 'ref.h5ad'
[2025-08-26T17:14:22,155] [main] INFO : Moving file to 'heart_og.n5'
[2025-08-26T17:14:22,169] [main] INFO : Done.
[2025-08-26T17:14:22,752] [main] INFO : Log level set to: INFO
[2025-08-26T17:14:23,048] [main] INFO : Saving in file 'query.h5ad'
[2025-08-26T17:14:23,268] [main] INFO : Moving file to 'heart_og.n5'
[2025-08-26T17:14:23,292] [main] INFO : Done.


In [44]:
st.align_pairs(container="heart_og.n5", num_genes=15, rendering_factor=0.5, max_epsilon=100, min_num_inliers_gene=30)

[2025-08-26T17:14:28,637] [main] INFO : Log level set to: INFO
[2025-08-26T17:14:28,788] [main] WARN : No input datasets specified. Trying to open all datasets in 'heart_og.n5' ...
[2025-08-26T17:14:28,789] [main] INFO : Opening 2 datasets
[2025-08-26T17:14:29,109] [main] INFO : Retrieving standard deviation of genes for all sections
[2025-08-26T17:14:29,110] [main] INFO : Computing standard deviation of genes for ref.h5ad (may take a while)
[2025-08-26T17:14:29,360] [main] INFO : Computing standard deviation of genes for query.h5ad (may take a while)
[2025-08-26T17:14:29,449] [main] INFO : Aligning ref.h5ad <> query.h5ad on 15 genes (32 threads)

[                                                  ] 0.60%
[                                                  ] 1.20%
[                                                  ] 1.80%
[=                                                 ] 2.40%
[=                                                 ] 3.00%
[=                                               

In [45]:
st.bdv_view3d(input="heart.n5", genes=['COL3A1,RSPO3,MYOZ2'], rendering_factor=0.5, z_spacing_factor=2)
st.align_global(container="heart.n5", absolute_threshold=100, rendering_factor=0.5, lmbda=0.0, skip_icp=True)


[2025-08-26T17:16:11,519] [main] INFO : Log level set to: INFO
[2025-08-26T17:16:11,662] [main] INFO : Opening all datasets in 'heart.n5' ...
[2025-08-26T17:16:11,772] [main] WARN : Anndata file seems to be missing some metadata. Trying to read it anyways...
[2025-08-26T17:16:11,869] [main] WARN : Anndata file seems to be missing some metadata. Trying to read it anyways...
Keymap list file /home/adyprat/.config/bigdataviewer/keymaps/keymaps.yaml not found. Using builtin styles.
Appearance settings file /home/adyprat/.config/bigdataviewer/appearance.yaml not found. Using defaults.
[2025-08-26T17:16:18,242] [main] INFO : Log level set to: INFO
[2025-08-26T17:16:18,381] [main] WARN : No input datasets specified. Trying to open all datasets in 'heart.n5' ...
[2025-08-26T17:16:18,520] [main] WARN : Anndata file seems to be missing some metadata. Trying to read it anyways...
[2025-08-26T17:16:18,651] [main] WARN : Anndata file seems to be missing some metadata. Trying to read it anyways...
i

In [84]:
import h5py, pathlib, anndata as ad

src = pathlib.Path("/mnt/e/git-workspace/CAST/demo/heart.n5/ref.h5ad")
dst = src.with_suffix(".ref_sanitized.h5ad")

with h5py.File(src, "r") as fsrc, h5py.File(dst, "w") as fdst:
    for k in fsrc.keys():
        if k != "__DATA_TYPES__":
            fsrc.copy(k, fdst)

adata = ad.read_h5ad(str(dst))

src = pathlib.Path("/mnt/e/git-workspace/CAST/demo/heart.n5/query.h5ad")
dst = src.with_suffix(".query_sanitized.h5ad")

with h5py.File(src, "r") as fsrc, h5py.File(dst, "w") as fdst:
    for k in fsrc.keys():
        if k != "__DATA_TYPES__":
            fsrc.copy(k, fdst)

adata = ad.read_h5ad(str(dst))


In [5]:
import anndata as ad
import numpy as np
queryAD = ad.read_h5ad('heart.n5/query.query_sanitized.h5ad')
queryAD.var.columns = ['stddev']
queryAD

AnnData object with n_obs × n_vars = 3184 × 67
    var: 'stddev'
    uns: 'model_sift', 'transform'
    obsm: 'spatial'

In [6]:
transform_matrix = queryAD.uns['transform'].reshape(2, 3)
transform_matrix = np.concatenate(
            (transform_matrix, np.array([0, 0, 1]).reshape(1, 3))
        )
print(transform_matrix)
locations = queryAD.obsm['spatial'].T
num_locations = locations.shape[1]
print(num_locations)
locations = np.concatenate(
            (locations, np.ones(num_locations).reshape(1, num_locations))
        )
print(locations.shape)
import numpy as np
# Apply affine transform in transform_matrix to locations
transformed = np.dot(transform_matrix, locations)
queryAD.obsm['X_STIM_spatial'] = transformed[:2, :].T


[[ 9.88330452e-01 -3.26632530e-02  1.94434558e+01]
 [-2.87393298e-03  1.01891749e+00  3.53470498e-01]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
3184
(3, 3184)


In [12]:
cell_types = ['Smooth muscle cells', 'Fibroblast', 'Atrial cardiomyocytes', 
                         'Cardiomyocytes', 'Endothelium', 'Epicardium', 
                         'Schwan progenitors', 'Ventricular cardiomyocytes']
cell_types_percentage = [x+'_percentage' for x in cell_types]

In [22]:
import scanpy as sc
queryAD_input = sc.read_h5ad('./query_input.h5ad')
queryAD_input

AnnData object with n_obs × n_vars = 3184 × 67
    obs: 'spot_x', 'spot_y', 'cell_count', 'Smooth muscle cells_percentage', 'Fibroblast_percentage', 'Atrial cardiomyocytes_percentage', 'Cardiomyocytes_percentage', 'Endothelium_percentage', 'Epicardium_percentage', 'Schwan progenitors_percentage', 'Ventricular cardiomyocytes_percentage', 'Cell_Num', 'New_X', 'New_Y', 'CellType'
    obsm: 'X_spatial', 'spatial'

In [33]:
refAD = sc.read_h5ad('./ref_input.h5ad')
refAD.obsm['X_STIM_spatial'] = refAD.obs[['New_X','New_Y']].values
refAD.write_h5ad('./ref_STIM.h5ad')
refAD

AnnData object with n_obs × n_vars = 3801 × 67
    obs: 'spot_x', 'spot_y', 'cell_count', 'Endothelium_percentage', 'Atrial cardiomyocytes_percentage', 'Cardiomyocytes_percentage', 'Epicardium_percentage', 'Fibroblast_percentage', 'Schwan progenitors_percentage', 'Smooth muscle cells_percentage', 'Ventricular cardiomyocytes_percentage', 'Cell_Num', 'New_X', 'New_Y', 'CellType'
    obsm: 'X_spatial', 'spatial', 'X_STIM_spatial'

In [23]:
queryAD.obsm['spatial']

array([[ 6.57516956e+02, -1.33918441e-01],
       [ 6.74407621e+02, -2.63715812e-01],
       [ 6.92380183e+02, -3.77012426e-01],
       ...,
       [ 4.93182212e+02,  9.70083014e+02],
       [ 5.10231735e+02,  9.69973623e+02],
       [ 5.28187471e+02,  9.69816582e+02]])

In [29]:
(queryAD_input.obs[['New_X','New_Y']].values == queryAD.obsm['spatial']).sum()/2

3184.0

In [30]:
queryAD.obs = queryAD_input.obs.copy()

In [31]:
queryAD.write_h5ad('./query_STIM.h5ad')

In [1]:
queryAD

NameError: name 'queryAD' is not defined